# RUKOPYS VLM Formula-Only Evaluation (Gold Train)

This notebook uses **ground-truth metadata** from the gold train split (formula only).

Pipeline:
- Use metadata `regions` (formula only) as fixed bboxes.
- Crop each formula region and run Qwen3-VL OCR.
- Build `submission.csv`-style output and compute a custom formula-text score.

Mount the same Qwen base model, Stage 2 LoRA output, RUKOPYS dataset, and the official metric notebook before running.

In [ ]:
INSTALL_DEPS = True

if INSTALL_DEPS:
    import subprocess
    import sys

    commands = [
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade-strategy",
            "only-if-needed",
            "accelerate",
            "peft",
            "bitsandbytes",
            "qwen-vl-utils",
            "pandas==2.2.2",
            "pillow<12",
        ],
        [sys.executable, "-m", "pip", "install", "-q", "-U", "git+https://github.com/huggingface/transformers.git"],
    ]
    for cmd in commands:
        print("Running:", " ".join(cmd), flush=True)
        subprocess.check_call(cmd)


In [ ]:
import gc
import json
import logging
import math
import os
import re
import shutil
import subprocess
import sys
import time
import warnings
from pathlib import Path

import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm

Image.MAX_IMAGE_PIXELS = None
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"


def suppress_transformers_noise():
    message = r".*Kwargs passed to `processor\.__call__` have to be in `processor_kwargs` dict.*"
    warnings.filterwarnings("ignore", message=message)
    logging.getLogger("transformers").setLevel(logging.ERROR)
    logging.getLogger("transformers.processing_utils").setLevel(logging.ERROR)
    try:
        from transformers.utils import logging as hf_logging

        hf_logging.set_verbosity_error()
    except Exception:
        pass


suppress_transformers_noise()

BASE_MODEL_CANDIDATES = [
    "/kaggle/input/models/qwen-lm/qwen-3-vl/transformers/8b-instruct/1",
    "/kaggle/input/qwen3-vl-8b-instruct",
    "Qwen/Qwen3-VL-8B-Instruct",
]

LORA_CANDIDATES = [
    "/kaggle/input/notebooks/notpitomon/htd-ocr-formula-vlm-qwen3vl-gold-fine-tune/qwen3vl_rukopys_stage2_gold/qwen3vl_rukopys_lora_final",
    "/kaggle/input/qwen3vl-rukopys-curriculum/qwen3vl_rukopys_lora_final",
    "/kaggle/input/qwen3vl-rukopys-lora/qwen3vl_rukopys_lora_final",
]

# Expected structure:
#   DATASET_ROOT/train/images/*.jpg
#   DATASET_ROOT/train/metadata.jsonl
DATASET_ROOT = "/kaggle/input/datasets/quii29/rukopys-dataset"

VALIDATION_RECORDS_CANDIDATES = [
    "/kaggle/working/qwen3vl_rukopys_curriculum/gold_validation_records.jsonl",
    "/kaggle/input/qwen3vl-rukopys-curriculum/gold_validation_records.jsonl",
]

RUN_SPLIT = "test"  # choices: "train", "test", "validation"
OUTPUT_CSV = "submission_formula_updated.csv"
TEST_MODE = False

# OCR settings.
CROP_OCR_MODE = "all_text"  # choices: "none", "smart", "all_text"
CROP_BATCH_SIZE = 2
CHECKPOINT_EVERY = 10
PROGRESS_LOG_EVERY = 1  # reliable Kaggle log line after every processed image
PROGRESS_BAR_WIDTH = 20
USE_TQDM_PROGRESS = False  # multiprocessing tqdm is often hidden in Kaggle logs
RESUME_PARTIALS = True
HYBRID_PARTIAL_PREFIX = "hybrid_prompt_v2_partial_results_gpu"
CHECKPOINT_INPUT_DIR = ""  # optional: set to a Kaggle input folder containing partial CSVs

MAX_PIXELS_CROP = 262_144
MAX_NEW_TOKENS_CROP = 192
CROP_PAD_RATIO = 0.04
LOAD_LORA_CROP_PROMPTS = False

SPECIAL_TEXT_MARKER_RULES = (
    "Use these special markers only when they are visible in the crop: "
    "~~word~~ for strikethrough text, ~~old~~{new} for strikethrough text with a visible correction, "
    "and [illegible] for an unreadable word inside an otherwise legible line. "
)

DOCUMENT_CONTEXT_RULES = (
    "The crop may come from Ukrainian dictation handwriting, historical Ukrainian/Cyrillic documents, "
    "school homework, exams, tables, formulas, chemistry notation, teacher marks, or mixed handwriting/print. "
    "Read only visible characters. Do not complete from canonical or memorized text. "
    "Preserve old spelling and do not modernize. "
)

FORMULA_PROMPT = (
    "Read this standalone math, logic, vector, matrix, determinant, set/relation, statistics, physics, "
    "or chemistry expression exactly as written. Return only formula text, using LaTeX when it is the "
    "clearest representation and plain Unicode when it better matches the handwriting. Do not wrap the "
    "answer in dollar signs. Preserve visible symbols, indices, superscripts, subscripts, arrows, fractions, "
    "matrix/determinant structure, punctuation, numbering, and strikethrough/correction markers. "
    + SPECIAL_TEXT_MARKER_RULES
    + "Do not solve, simplify, normalize, explain, or convert old notation into a different style. "
    + DOCUMENT_CONTEXT_RULES
    + "Return only the transcription. No JSON, no Markdown, no explanation."
 )

CROP_PROMPTS = {
    "formula": FORMULA_PROMPT,
}

VALID_TYPES = {"formula", "handwritten", "printed", "table", "diagram", "table_with_formulas"}
TEXT_TYPES = {"formula"}
IMAGE_EXTENSIONS = [".png", ".jpg", ".jpeg", ".webp", ".bmp"]

In [ ]:
def find_model_id():
    for item in BASE_MODEL_CANDIDATES:
        if item.startswith("/") and Path(item).exists():
            return item
        if not item.startswith("/"):
            return item
    raise FileNotFoundError("No base model found. Add Qwen3-VL to Kaggle input or enable internet.")


def find_lora_dir():
    for item in LORA_CANDIDATES:
        p = Path(item)
        if (p / "adapter_config.json").exists():
            return p
    for root, _, files in os.walk("/kaggle/input"):
        if "adapter_config.json" in files:
            return Path(root)
    raise FileNotFoundError("No LoRA adapter_config.json found. Add the Stage 2 training notebook output as Kaggle input.")


def get_dataset_root():
    root = Path(DATASET_ROOT)
    required_split = "test" if RUN_SPLIT == "test" else "train"
    metadata_path = root / required_split / "metadata.jsonl"
    if not metadata_path.exists():
        raise FileNotFoundError(
            f"DATASET_ROOT is not configured correctly: {root}. Expected {required_split}/metadata.jsonl under this path."
        )
    return root


def find_validation_records_path(lora_path):
    candidates = [Path(p) for p in VALIDATION_RECORDS_CANDIDATES]
    candidates.append(Path(lora_path).parent / "gold_validation_records.jsonl")
    candidates.append(Path(lora_path).parent.parent / "gold_validation_records.jsonl")
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError("No gold_validation_records.jsonl found. Set RUN_SPLIT='test' or mount the validation records.")


def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def resolve_image_path(root, split, file_name):
    raw = Path(file_name)
    name = raw.name
    stem = raw.stem
    candidate_names = [name] + [stem + ext for ext in IMAGE_EXTENSIONS if stem + ext != name]
    candidates = [root / split / file_name, root / file_name]
    for candidate_name in candidate_names:
        candidates.extend([root / split / "images" / candidate_name, root / split / candidate_name])
    for p in candidates:
        if p.exists():
            return str(p)
    return str(root / split / "images" / candidate_names[0])


def load_prompt_config(lora_dir):
    global CROP_PROMPTS, MAX_PIXELS_CROP
    cfg_path = Path(lora_dir) / "rukopys_prompt_config.json"
    if not cfg_path.exists():
        return
    cfg = json.loads(cfg_path.read_text(encoding="utf-8"))
    if LOAD_LORA_CROP_PROMPTS:
        CROP_PROMPTS.update(cfg.get("crop_prompts", {}))
    MAX_PIXELS_CROP = int(cfg.get("max_pixels_crop", MAX_PIXELS_CROP))


model_id = find_model_id()
lora_dir = find_lora_dir()
dataset_root = get_dataset_root()
load_prompt_config(lora_dir)

if RUN_SPLIT == "validation":
    validation_path = find_validation_records_path(lora_dir)
    test_records = read_jsonl(validation_path)
    IMAGE_SPLIT = "train"
    OUTPUT_CSV = "hybrid_validation_pred.csv"
    print("Validation records:", validation_path)
elif RUN_SPLIT == "train":
    test_records = read_jsonl(dataset_root / "train" / "metadata.jsonl")
    IMAGE_SPLIT = "train"
    OUTPUT_CSV = OUTPUT_CSV or "hybrid_gold_train_formula.csv"
else:
    import json
    df_sub = pd.read_csv("/kaggle/input/datasets/notpitomon/htd-submit/0.81122.csv")
    # Right here
    test_records = []
    for idx, row in df_sub.iterrows():
        regs = json.loads(row.regions) if isinstance(row.regions, str) else []
        test_records.append({"file_name": row.image, "regions": regs})
    IMAGE_SPLIT = "test"
    OUTPUT_CSV = OUTPUT_CSV or "submission.csv"

if TEST_MODE:
    test_records = test_records[:4]

print("Base model:", model_id)
print("LoRA:", lora_dir)
print("Dataset:", dataset_root)
print("Run split:", RUN_SPLIT)
print("Images:", len(test_records))

In [ ]:
checkpoint_dir = Path(CHECKPOINT_INPUT_DIR) if CHECKPOINT_INPUT_DIR else None
if checkpoint_dir and checkpoint_dir.exists():
    print("Restoring hybrid checkpoints from", checkpoint_dir)
    for src in checkpoint_dir.glob(f"{HYBRID_PARTIAL_PREFIX}*.csv"):
        dst = Path("/kaggle/working") / src.name
        shutil.copy2(src, dst)
        print(f"Copied {src} -> {dst}")
else:
    print("No hybrid checkpoint input configured.")

print("Current hybrid checkpoint files:")
for p in sorted(Path("/kaggle/working").glob(f"{HYBRID_PARTIAL_PREFIX}*.csv")):
    try:
        rows = len(pd.read_csv(p).drop_duplicates(subset=["image"], keep="last"))
    except Exception:
        rows = "?"
    print(f"{p} size={p.stat().st_size} rows={rows}")


In [ ]:
def clamp_xyxy(box, width, height):
    if not isinstance(box, (list, tuple)) or len(box) != 4:
        return None
    try:
        x1, y1, x2, y2 = [float(v) for v in box]
    except Exception:
        return None
    x1, x2 = sorted((max(0, min(width, x1)), max(0, min(width, x2))))
    y1, y2 = sorted((max(0, min(height, y1)), max(0, min(height, y2))))
    if x2 - x1 < 3 or y2 - y1 < 3:
        return None
    return [int(round(x1)), int(round(y1)), int(round(x2)), int(round(y2))]


def sort_regions(regions):
    return sorted(regions, key=lambda r: (r["bbox"][1], r["bbox"][0]))


def strip_internal_fields(region):
    return {"bbox": region["bbox"], "type": region.get("type", "formula"), "text": str(region.get("text") or "")}


def clean_crop_text(text):
    text = (text or "").strip()
    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z]*", "", text).strip()
        text = re.sub(r"```$", "", text).strip()
    text = re.sub(r"^(text|transcription|answer)\s*:\s*", "", text, flags=re.I).strip()
    if len(text) >= 2 and text[0] == text[-1] and text[0] in {"'", '"'}:
        text = text[1:-1].strip()
    if text.startswith("[") or text.startswith("{"):
        try:
            obj = json.loads(text)
            if isinstance(obj, dict) and "text" in obj:
                text = str(obj["text"])
            else:
                return ""
        except Exception:
            return ""
    return text[:500]

In [ ]:
def get_record_image_size(record, image_path):
    w = int(record.get("image_width") or 0)
    h = int(record.get("image_height") or 0)
    if w > 0 and h > 0:
        return w, h
    with Image.open(image_path) as img:
        return img.size


def is_formula_region(region):
    return str(region.get("type") or "").strip().lower() == "formula"


def build_formula_regions_from_record(record, image_path):
    w, h = get_record_image_size(record, image_path)
    regions = []
    for r in record.get("regions") or []:
        box = clamp_xyxy(r.get("bbox"), w, h)
        if box is None:
            continue
        regions.append({
            "bbox": box, 
            "type": r.get("type", "formula"), 
            "text": str(r.get("text") or "")
        })
    return sort_regions(regions)

In [ ]:
from peft import PeftModel
from qwen_vl_utils import process_vision_info
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig


def configure_processor_for_generation(processor):
    if processor.tokenizer.pad_token_id is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token
    processor.tokenizer.padding_side = "left"
    return processor


def load_qwen_model(device):
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )
    base = AutoModelForImageTextToText.from_pretrained(
        model_id,
        device_map={"": device},
        quantization_config=quantization_config,
        dtype=torch.float16,
        trust_remote_code=True,
        attn_implementation="sdpa",
        low_cpu_mem_usage=True,
    )
    model = PeftModel.from_pretrained(base, str(lora_dir))
    model.eval()
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    processor = configure_processor_for_generation(processor)
    if processor.tokenizer.pad_token_id is not None:
        model.generation_config.pad_token_id = processor.tokenizer.pad_token_id
    return model, processor


def apply_chat_template(processor, messages):
    candidates = [
        {"tokenize": False, "add_generation_prompt": True, "template_kwargs": {"enable_thinking": False}},
        {"tokenize": False, "add_generation_prompt": True, "processor_kwargs": {"enable_thinking": False}},
        {"tokenize": False, "add_generation_prompt": True, "enable_thinking": False},
        {"tokenize": False, "add_generation_prompt": True},
    ]
    for kwargs in candidates:
        try:
            with warnings.catch_warnings():
                warnings.filterwarnings(
                    "ignore",
                    message=r".*Kwargs passed to `processor\.__call__` have to be in `processor_kwargs` dict.*",
                )
                return processor.apply_chat_template(messages, **kwargs)
        except TypeError:
            continue
    return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def generate_batch(model, processor, messages_batch, device, max_new_tokens):
    processor.tokenizer.padding_side = "left"
    texts = [apply_chat_template(processor, m) for m in messages_batch]
    image_inputs, video_inputs = process_vision_info(messages_batch)
    try:
        inputs = processor(
            text=texts,
            images=image_inputs,
            videos=video_inputs,
            text_kwargs={"padding": True, "return_tensors": "pt"},
            images_kwargs={"return_tensors": "pt"},
            videos_kwargs={"return_tensors": "pt"},
        )
    except TypeError:
        inputs = processor(
            text=texts,
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )
    inputs = inputs.to(device)
    with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.float16):
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
        )
    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, out)]
    decoded = processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    del inputs, out, trimmed
    torch.cuda.empty_cache()
    return decoded


In [ ]:
def resize_to_pixel_budget(img, max_pixels):
    w, h = img.size
    total = max(1, w * h)
    if total <= max_pixels:
        return img
    scale = (max_pixels / total) ** 0.5
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))
    return img.resize((new_w, new_h), Image.Resampling.LANCZOS)


def crop_image(image_path, bbox):
    with Image.open(image_path) as img:
        img = img.convert("RGB")
        w, h = img.size
        x1, y1, x2, y2 = bbox
        pad = int(round(max(x2 - x1, y2 - y1) * CROP_PAD_RATIO))
        x1 = max(0, x1 - pad)
        y1 = max(0, y1 - pad)
        x2 = min(w, x2 + pad)
        y2 = min(h, y2 + pad)
        return resize_to_pixel_budget(img.crop((x1, y1, x2, y2)), MAX_PIXELS_CROP)


def should_crop_ocr(region):
    if CROP_OCR_MODE == "none":
        return False
    if region.get("type") not in TEXT_TYPES:
        return False
    if CROP_OCR_MODE == "all_text":
        return True
    text = str(region.get("text") or "")
    return (not text.strip()) or len(text) < 4 or len(text) > 160


def crop_messages(image_path, region):
    prompt = CROP_PROMPTS["formula"]
    return [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": crop_image(image_path, region["bbox"])},
                {"type": "text", "text": prompt},
            ],
        }
]


def ocr_regions(qwen_model, processor, image_path, regions, device):
    crop_indices = [i for i, r in enumerate(regions) if should_crop_ocr(r)]
    for start in range(0, len(crop_indices), CROP_BATCH_SIZE):
        batch_indices = crop_indices[start:start + CROP_BATCH_SIZE]
        msgs = [crop_messages(image_path, regions[i]) for i in batch_indices]
        try:
            outs = generate_batch(qwen_model, processor, msgs, device, MAX_NEW_TOKENS_CROP)
        except Exception as e:
            print("Crop OCR batch failed:", e, flush=True)
            torch.cuda.empty_cache()
            gc.collect()
            continue
        for idx, out in zip(batch_indices, outs):
            text = clean_crop_text(out)
            if text:
                regions[idx]["text"] = text
    return sort_regions([strip_internal_fields(r) for r in regions])


def infer_one_image(qwen_model, processor, image_path, record, device):
    regions = build_formula_regions_from_record(record, image_path)
    if not regions:
        return []
    regions = ocr_regions(qwen_model, processor, image_path, regions, device)
    return sort_regions(regions)

In [ ]:
import multiprocessing as mp


def format_duration(seconds):
    if seconds is None or seconds <= 0:
        return "--:--"
    seconds = int(round(seconds))
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    secs = seconds % 60
    if hours:
        return f"{hours:02d}:{minutes:02d}:{secs:02d}"
    return f"{minutes:02d}:{secs:02d}"


def progress_bar(done, total, width=PROGRESS_BAR_WIDTH):
    ratio = done / max(1, total)
    filled = min(width, max(0, int(round(width * ratio))))
    return "█" * filled + " " * (width - filled)


def print_progress_line(gpu_id, done, total, run_done, elapsed, region_count, image_name):
    pct = 100.0 * done / max(1, total)
    speed = run_done / elapsed if run_done > 0 and elapsed > 0 else 0.0
    sec_per_img = elapsed / run_done if run_done > 0 else 0.0
    remaining = max(0, total - done)
    eta = remaining / speed if speed > 0 else None
    bar = progress_bar(done, total)
    last = str(image_name or "")[:18]
    if speed > 0:
        timing = f"{format_duration(elapsed)}<{format_duration(eta)}, {sec_per_img:.2f}s/img, speed={speed:.2f} img/s"
    else:
        timing = f"resume, speed=-- img/s"
    print(
        f"GPU {gpu_id}: {pct:3.0f}%|{bar}| {done}/{total} "
        f"[{timing}, regions={region_count}, last={last}]",
        flush=True,
    )


def worker_process(gpu_id, task_queue, total_records, output_csv, initial_done):
    suppress_transformers_noise()
    device = f"cuda:{gpu_id}"
    print(f"[GPU {gpu_id}] loading Qwen for remaining images", flush=True)
    qwen_model, processor = load_qwen_model(device)
    print(f"[GPU {gpu_id}] model loaded; starting formula OCR", flush=True)

    done = set(initial_done)
    results = []
    if RESUME_PARTIALS and Path(output_csv).exists():
        try:
            old = pd.read_csv(output_csv)
            old = old.drop_duplicates(subset=["image"], keep="last")
            results = old.to_dict("records")
            print(f"[GPU {gpu_id}] loading {len(results)} existing rows from {output_csv}", flush=True)
        except Exception as e:
            print(f"[GPU {gpu_id}] could not read checkpoint: {e}", flush=True)

    print_progress_line(gpu_id, len(done), total_records, 0, 0.0, 0, "resumed" if done else "start")

    # In dynamic mode, using tqdm isn't straightforward without a shared counter,
    # so we rely primarily on print_progress_line.
    run_start = time.time()
    run_done = 0

    while True:
        try:
            # timeout just in case it hangs
            rec = task_queue.get(timeout=10)
        except Exception:
            break
            
        if rec is None:
            break

        image_name = Path(rec["file_name"]).name
        # Even if initial_done was populated, another GPU might have completed it if it wasn't filtered,
        # but we already filter before enqueuing. Checking just in case we resume locally.
        if image_name in done:
            continue
            
        image_path = resolve_image_path(dataset_root, IMAGE_SPLIT, rec["file_name"])
        try:
            regions = infer_one_image(qwen_model, processor, image_path, rec, device)
        except Exception as e:
            print(f"[GPU {gpu_id}] failed {image_name}: {e}", flush=True)
            regions = []
            torch.cuda.empty_cache()
            gc.collect()
            
        results.append({"image": image_name, "regions": json.dumps(regions, ensure_ascii=False)})
        done.add(image_name)
        run_done += 1
        elapsed = max(1e-6, time.time() - run_start)
        
        if run_done % PROGRESS_LOG_EVERY == 0:
            print_progress_line(gpu_id, len(done), total_records, run_done, elapsed, len(regions), image_name)

        if len(results) % CHECKPOINT_EVERY == 0:
            tmp = output_csv + ".tmp"
            pd.DataFrame(results).drop_duplicates(subset=["image"], keep="last").to_csv(tmp, index=False)
            os.replace(tmp, output_csv)

    tmp = output_csv + ".tmp"
    pd.DataFrame(results).drop_duplicates(subset=["image"], keep="last").to_csv(tmp, index=False)
    os.replace(tmp, output_csv)
    print(f"[GPU {gpu_id}] done {run_done} tasks; saved {output_csv}", flush=True)


def detect_num_gpus():
    try:
        out = subprocess.check_output(["nvidia-smi", "-L"]).decode("utf-8").strip()
        return max(1, len([x for x in out.splitlines() if x.strip()]))
    except Exception:
        return max(1, torch.cuda.device_count())


mp.set_start_method("fork", force=True)
num_gpus = detect_num_gpus()
print("GPUs:", num_gpus, flush=True)

global_done = set()
if RESUME_PARTIALS:
    for gpu_id in range(num_gpus):
        path = f"{HYBRID_PARTIAL_PREFIX}{gpu_id}.csv"
        if Path(path).exists():
            try:
                df = pd.read_csv(path)
                global_done.update(df["image"].tolist())
            except Exception:
                pass
    print(f"Resumed {len(global_done)} previously processed images across all GPUs.", flush=True)

# Build the queue dynamically
task_queue = mp.Queue()
pending_count = 0
for rec in test_records:
    image_name = Path(rec["file_name"]).name
    if image_name not in global_done:
        task_queue.put(rec)
        pending_count += 1

# Put sentinels so workers can exit
for _ in range(num_gpus):
    task_queue.put(None)

print(f"Pending images to process: {pending_count} out of {len(test_records)}", flush=True)

processes = []
partials = []
for gpu_id in range(num_gpus):
    output_csv = f"{HYBRID_PARTIAL_PREFIX}{gpu_id}.csv"
    partials.append(output_csv)
    p = mp.Process(target=worker_process, args=(gpu_id, task_queue, len(test_records), output_csv, global_done))
    p.start()
    processes.append(p)

for p in processes:
    p.join()

bad_exitcodes = [p.exitcode for p in processes if p.exitcode not in (0, None)]
if bad_exitcodes:
    raise RuntimeError(f"One or more workers failed with exit codes: {bad_exitcodes}")

frames = []
for path in partials:
    if Path(path).exists():
        frame = pd.read_csv(path)
        print(f"Partial {path}: rows={len(frame)}", flush=True)
        frames.append(frame)
if not frames:
    raise RuntimeError("No partial outputs were created.")

final = pd.concat(frames, ignore_index=True).drop_duplicates(subset=["image"], keep="last")
order = [Path(r["file_name"]).name for r in test_records]
final = final.set_index("image").reindex(order).reset_index()
final["regions"] = final["regions"].fillna("[]")
final.to_csv(OUTPUT_CSV, index=False)
print("Wrote", OUTPUT_CSV, "rows=", len(final), flush=True)
final.head()

In [ ]:
df = pd.read_csv(OUTPUT_CSV)
assert list(df.columns) == ["image", "regions"], df.columns
assert len(df) == len(test_records), (len(df), len(test_records))

bad = []
region_counts = []
for row in df.itertuples(index=False):
    try:
        parsed = json.loads(row.regions)
        assert isinstance(parsed, list)
        region_counts.append(len(parsed))
        for item in parsed:
            assert "bbox" in item and "type" in item and "text" in item
            assert isinstance(item["bbox"], list) and len(item["bbox"]) == 4
            assert item["type"] in VALID_TYPES
    except Exception as e:
        bad.append((row.image, str(e)))
        if len(bad) >= 5:
            break

print("Bad rows:", bad[:5])
print("Images:", len(df))
print("Total regions:", sum(region_counts))
print("Avg regions/page:", round(sum(region_counts) / max(1, len(region_counts)), 2))
print("Ready:", OUTPUT_CSV)


def filter_formula_regions(regions, keep_meta=False):
    out = []
    for r in regions or []:
        rtype = str(r.get("type") or "").strip().lower()
        if rtype != "formula":
            continue
        item = {
            "bbox": r.get("bbox"),
            "type": "formula",
            "text": str(r.get("text") or ""),
        }
        if keep_meta:
            if "language" in r:
                item["language"] = r.get("language")
            if "legibility" in r:
                item["legibility"] = r.get("legibility")
        out.append(item)
    return out


def normalize_pred_df(pred_df):
    rows = []
    for row in pred_df.itertuples(index=False):
        try:
            parsed = json.loads(row.regions)
        except Exception:
            parsed = []
        filtered = filter_formula_regions(parsed, keep_meta=False)
        rows.append({
            "image": row.image,
            "regions": json.dumps(filtered, ensure_ascii=False),
        })
    return pd.DataFrame(rows)


def build_solution_df(records):
    rows = []
    for rec in records:
        regions = filter_formula_regions(rec.get("regions") or [], keep_meta=True)
        rows.append({
            "image": Path(rec["file_name"]).name,
            "regions": json.dumps(regions, ensure_ascii=False),
        })
    return pd.DataFrame(rows)


def normalize_formula_text(text):
    text = "" if text is None else str(text)
    text = text.strip()
    if len(text) >= 2 and text[0] == "$" and text[-1] == "$":
        text = text[1:-1]
    text = re.sub(r"\s+", "", text)
    text = re.sub(r"[\u2010-\u2015\u2212\uFE58\uFE63\uFF0D]", "-", text)
    return text


def levenshtein(s1, s2):
    if len(s1) < len(s2):
        return levenshtein(s2, s1)
    if len(s2) == 0:
        return len(s1)
    prev = list(range(len(s2) + 1))
    for i, c1 in enumerate(s1):
        curr = [i + 1]
        for j, c2 in enumerate(s2):
            curr.append(min(
                prev[j + 1] + 1,
                curr[j] + 1,
                prev[j] + (c1 != c2),
            ))
        prev = curr
    return prev[-1]


def formula_text_score(solution_df, pred_df, w_char=0.8, w_exact=0.2):
    pred_lookup = {
        row.image: json.loads(row.regions) if isinstance(row.regions, str) else []
        for row in pred_df.itertuples(index=False)
    }
    total_score = 0.0
    total_char = 0.0
    total_exact = 0.0
    total_regions = 0
    for row in solution_df.itertuples(index=False):
        gt_regions = json.loads(row.regions) if isinstance(row.regions, str) else []
        pred_regions = pred_lookup.get(row.image, [])
        pred_map = {tuple(r.get("bbox") or []): r.get("text", "") for r in pred_regions}
        for gt in gt_regions:
            gt_text = gt.get("text", "")
            pred_text = pred_map.get(tuple(gt.get("bbox") or []), "")
            gt_norm = normalize_formula_text(gt_text)
            pred_norm = normalize_formula_text(pred_text)
            if not gt_norm and not pred_norm:
                char_sim = 1.0
                exact = 1.0
            elif not gt_norm:
                char_sim = 0.0
                exact = 0.0
            else:
                dist = levenshtein(pred_norm, gt_norm)
                cer = dist / max(len(gt_norm), 1)
                char_sim = max(0.0, 1.0 - cer)
                exact = 1.0 if pred_norm == gt_norm else 0.0
            score = w_char * char_sim + w_exact * exact
            total_score += score
            total_char += char_sim
            total_exact += exact
            total_regions += 1
    if total_regions == 0:
        return {"formula_score": 0.0, "char_similarity": 0.0, "exact_match": 0.0, "n_regions": 0, "n_images": len(solution_df)}
    return {
        "formula_score": total_score / total_regions,
        "char_similarity": total_char / total_regions,
        "exact_match": total_exact / total_regions,
        "n_regions": total_regions,
        "n_images": len(solution_df),
    }


solution_df = build_solution_df(test_records)
pred_df = normalize_pred_df(df)
report = formula_text_score(solution_df, pred_df, w_char=0.8, w_exact=0.2)

print("Formula text score (%):", round(report["formula_score"] * 100, 2))
print("Char similarity (%):", round(report["char_similarity"] * 100, 2))
print("Exact match (%):", round(report["exact_match"] * 100, 2))
print("Regions:", report["n_regions"], "Images:", report["n_images"])